# 📓 Notebook 5 — Integrando Datos Satelitales Reales al Videojuego

**Sección 3 — Proyecto Final** · Día 4 (tarde)

### Objetivo de hoy
Integrar los **datos satelitales reales** (NDVI / uso de suelo) que procesaron en la Sección 1 como **condición inicial** de su videojuego, elegir la zona y problema específico de su ciudad, y comenzar la personalización creativa del proyecto final (narrativa, estética, mecánicas únicas).

Esta es la sesión donde su videojuego deja de ser un prototipo genérico y se convierte en **el proyecto de su equipo**, sobre un lugar real.


## 0. Conectemos con lo anterior

En la Sección 1 calcularon el NDVI real de su zona de estudio. Hasta ahora (Notebooks 3 y 4), su videojuego usaba mapas **generados al azar**.

**✏️ Antes de cargar sus datos reales, respondan de memoria:**

1. Según lo que vieron en la Sección 1, ¿qué porcentaje aproximado de su zona es vegetación? ¿Y agua? ¿Y área construida?
2. ¿Su zona ha cambiado mucho entre las fechas que compararon, o se ha mantenido parecida? ¿Esperan que el mapa que genere este notebook se parezca al mapa aleatorio que usaron en el Notebook 4, o sea muy distinto?

*Vamos a comparar esto con los datos reales en la Sección 2.*


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import json, os, glob

TIPOS_CELDA = {
    0: {"nombre": "agua", "emoji": "💧", "color": "#3b82c4"},
    1: {"nombre": "vegetacion", "emoji": "🌳", "color": "#4caf6b"},
    2: {"nombre": "urbano", "emoji": "🏠", "color": "#9e9e9e"},
    3: {"nombre": "industrial", "emoji": "🏭", "color": "#e08a2c"},
}
CMAP = ListedColormap([TIPOS_CELDA[i]["color"] for i in range(len(TIPOS_CELDA))])

def dibujar_mapa(grid, titulo="Mapa"):
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    ax.imshow(grid, cmap=CMAP, vmin=0, vmax=len(TIPOS_CELDA) - 1)
    ax.set_title(titulo)
    ax.set_xticks(range(grid.shape[1])); ax.set_yticks(range(grid.shape[0]))
    ax.grid(color="white", linewidth=1)
    ax.tick_params(length=0, labelbottom=False, labelleft=False)
    plt.show()

print("Listo ✅ — vamos a cargar los datos reales de su zona.")


## 1. Checklist de integración de datos

Antes de cargar sus datos, revisen con su equipo que tengan lo siguiente **de la Sección 1**:

- [ ] Un archivo con su índice **NDVI** procesado (`.npy`, `.tif` o `.csv`) de su zona de estudio
- [ ] Saben las coordenadas / nombre de la zona que analizaron (ej. "Centro de Mérida")
- [ ] Tienen (idealmente) 2 fechas distintas para comparar antes/después
- [ ] Sabe cada quien en el equipo qué representa cada rango de NDVI (agua, suelo desnudo, vegetación baja/alta)

Si a su equipo le falta algún archivo, esta libreta trae una **zona de ejemplo simulada** para que puedan seguir avanzando mientras lo consiguen; solo reemplacen la celda de carga de datos cuando lo tengan.


## 2. Cargar sus datos NDVI reales

Suban a esta carpeta (en Colab: ícono de carpeta a la izquierda → arrastrar archivo) su archivo de NDVI procesado. Soportamos dos formatos simples:

- **`.npy`**: un arreglo de NumPy ya guardado (`np.save(...)` desde la Sección 1)
- **`.csv`**: una tabla de valores NDVI (filas = filas del mapa, columnas = columnas del mapa)

Escriban el nombre de su archivo en `NOMBRE_ARCHIVO_NDVI` abajo.


In [ ]:
# ✏️ EDITA: escribe el nombre de tu archivo NDVI de la Sección 1 (debe estar en esta misma carpeta)
NOMBRE_ARCHIVO_NDVI = "ndvi_mi_zona.npy"   # cambia esto por tu archivo real, ej. "ndvi_centro_merida.npy"

def cargar_ndvi_real(nombre_archivo):
    if not os.path.exists(nombre_archivo):
        return None
    if nombre_archivo.endswith(".npy"):
        return np.load(nombre_archivo)
    elif nombre_archivo.endswith(".csv"):
        return np.loadtxt(nombre_archivo, delimiter=",")
    else:
        raise ValueError("Formato no soportado — usa .npy o .csv")

def generar_ndvi_ejemplo(n=15, semilla=11):
    """NDVI simulado (rango tipico -1 a 1) para practicar si aun no tienen su archivo real."""
    rng = np.random.default_rng(semilla)
    base = rng.normal(loc=0.35, scale=0.25, size=(n, n))
    base[0:3, 0:3] = rng.normal(loc=-0.1, scale=0.05, size=(3, 3))  # zona de "agua" simulada
    return np.clip(base, -1, 1)

ndvi = cargar_ndvi_real(NOMBRE_ARCHIVO_NDVI)
if ndvi is None:
    print(f"⚠️ No se encontró '{NOMBRE_ARCHIVO_NDVI}'. Usando NDVI de ejemplo mientras consiguen su archivo real.")
    ndvi = generar_ndvi_ejemplo()
    usando_datos_reales = False
else:
    print(f"✅ NDVI real cargado desde '{NOMBRE_ARCHIVO_NDVI}' — forma: {ndvi.shape}")
    usando_datos_reales = True

plt.figure(figsize=(5.5, 5.5))
plt.imshow(ndvi, cmap="RdYlGn", vmin=-1, vmax=1)
plt.colorbar(label="NDVI")
plt.title("NDVI de tu zona" + ("" if usando_datos_reales else " (EJEMPLO)"))
plt.xticks([]); plt.yticks([])
plt.show()


### 🔍 Observen y reflexionen

**✏️ Comparen este mapa de NDVI real con su predicción de la Sección 0:**

- ¿El patrón de verdes (vegetación) y rojos/amarillos (menos vegetación) coincide con lo que esperaban de su zona? *(reemplazar)*
- Si están usando el NDVI de ejemplo (no el real todavía), ¿qué diferencia creen que habrá cuando lo reemplacen por el suyo? *(reemplazar)*


## 3. Convertir NDVI real a tipos de celda del videojuego

Usamos umbrales típicos de NDVI para clasificar cada celda en uno de los 4 tipos de nuestro juego. **Ajusten estos umbrales** si en la Sección 1 encontraron valores distintos para su zona específica.

| Rango NDVI | Tipo de celda |
|---|---|
| `< 0.1` | 💧 Agua / superficie sin vegetación |
| `0.1 – 0.25` | 🏠 Urbano / suelo desnudo |
| `0.25 – 0.45` | 🏭 Industrial / vegetación dispersa (ajustar con su criterio) |
| `> 0.45` | 🌳 Vegetación |

> Nota: NDVI por sí solo no distingue "urbano" de "industrial" — en la Sección 1 quizás ya combinaron NDVI con otra capa (uso de suelo). Si tienen esa clasificación más precisa, pueden reemplazar esta función por la suya.

**🤔 Antes de correr el código:** miren la nota de arriba otra vez. ¿Por qué creen que un solo número (NDVI) no puede distinguir "fábrica" de "casa"? ¿Qué otro dato (que no sea NDVI) les ayudaría a diferenciarlos?


In [ ]:
# ✏️ AJUSTA los umbrales si su equipo encontró rangos distintos en la Sección 1
UMBRAL_AGUA = 0.10
UMBRAL_URBANO = 0.25
UMBRAL_INDUSTRIAL = 0.45

def ndvi_a_grid(ndvi):
    grid = np.zeros_like(ndvi, dtype=int)
    grid[ndvi < UMBRAL_AGUA] = 0
    grid[(ndvi >= UMBRAL_AGUA) & (ndvi < UMBRAL_URBANO)] = 2
    grid[(ndvi >= UMBRAL_URBANO) & (ndvi < UMBRAL_INDUSTRIAL)] = 3
    grid[ndvi >= UMBRAL_INDUSTRIAL] = 1
    return grid

grid_inicial = ndvi_a_grid(ndvi)
dibujar_mapa(grid_inicial, "Mapa inicial del juego — derivado de NDVI real" if usando_datos_reales else "Mapa inicial (EJEMPLO)")

total = grid_inicial.size
print("Distribución inicial del mapa:")
for codigo, info in TIPOS_CELDA.items():
    pct = 100 * np.sum(grid_inicial == codigo) / total
    print(f"  {info['emoji']} {info['nombre']}: {pct:.1f}%")


### 🔍 Observen y reflexionen

**✏️ Comparen los porcentajes reales (impresos arriba) con su predicción de la Sección 0. Escriban su respuesta:**

- ¿Qué tan cerca estuvo su predicción de vegetación/agua/construcción? *(reemplazar)*
- Prueben mover `UMBRAL_URBANO` y `UMBRAL_INDUSTRIAL` un poco (por ejemplo, ±0.05) y vuelvan a correr la celda. ¿Cambia mucho la distribución? ¿Qué les dice eso sobre qué tan "sensible" es su clasificación a estos umbrales? *(reemplazar)*


## 4. Elijan su zona y problema específico

Cada equipo elige una **zona concreta** de su ciudad y un **problema específico** que quieren explorar con su videojuego. Completen esto junto con su mentor:


In [ ]:
# ✏️ EDITA esta ficha con la información de TU equipo

ficha_proyecto = {
    "equipo": "NOMBRE_DE_TU_EQUIPO",
    "zona_elegida": "ej. Colonia García Ginerés, Mérida",
    "problema_especifico": "ej. Pérdida de árboles urbanos por nuevas construcciones en la última década",
    "pregunta_que_explora_el_juego": "ej. ¿Qué pasaría si cada nueva vivienda exigiera plantar 2 árboles?",
    "fecha_ndvi_usada": "ej. Sentinel-2, marzo 2024",
}

for campo, valor in ficha_proyecto.items():
    print(f"{campo}: {valor}")


## 5. Guía de narrativa para el juego

Un buen videojuego educativo no es solo mecánicas — también cuenta una historia. Respondan estas preguntas con su equipo (pueden usar papel rotafolio / posticks / colores para hacer una lluvia de ideas antes de escribir aquí):

1. **Protagonista**: ¿quién es el jugador dentro del juego? (¿alcalde/alcaldesa? ¿un habitante cualquiera? ¿un dron satelital?)
2. **Meta**: ¿qué está tratando de lograr o evitar en su zona?
3. **Tensión central**: ¿cuál es el conflicto o disyuntiva principal? (ej. crecimiento económico vs. conservación)
4. **Tono/estética**: ¿su juego se siente serio y realista, o divertido y exagerado? ¿qué colores/estilo visual usarán?
5. **Nombre del juego**: denle un título a su proyecto.

**🔍 Antes de escribir:** revisen el `problema_especifico` que anotaron en la Sección 4 y la distribución real de su mapa (Sección 3). ¿Su narrativa refleja de verdad lo que encontraron en los datos, o se les está olvidando algo importante que vieron en las imágenes satelitales?


In [ ]:
# ✏️ EDITA con las respuestas de tu equipo

narrativa = {
    "nombre_del_juego": "________________",
    "protagonista": "________________",
    "meta": "________________",
    "tension_central": "________________",
    "tono_y_estetica": "________________",
}

for campo, valor in narrativa.items():
    print(f"{campo}: {valor}")


## 6. Guardar la versión alpha del proyecto final

Esta celda junta todo — mapa real derivado de NDVI, ficha del proyecto y narrativa — en un archivo `proyecto_final_alpha.json`. Este es el **producto esperado de hoy**: *"Versión alpha del proyecto final con datos reales integrados y narrativa definida."*


In [ ]:
proyecto_alpha = {
    "ficha_proyecto": ficha_proyecto,
    "narrativa": narrativa,
    "usando_datos_reales": usando_datos_reales,
    "grid_inicial": grid_inicial.tolist(),
    "distribucion_inicial_pct": {
        info["nombre"]: round(100 * np.sum(grid_inicial == codigo) / grid_inicial.size, 1)
        for codigo, info in TIPOS_CELDA.items()
    },
}

with open("proyecto_final_alpha.json", "w", encoding="utf-8") as f:
    json.dump(proyecto_alpha, f, ensure_ascii=False, indent=2)

print("✅ Versión alpha guardada en 'proyecto_final_alpha.json'")
print(f"   Datos reales integrados: {'Sí' if usando_datos_reales else 'No (usando ejemplo — reemplazar antes de la entrega final)'}")


## 7. Conectar con el Notebook 4 (juego con escenarios)

Para seguir jugando con su mapa real en vez del genérico, en el **Notebook 4** reemplacen la función `generar_mapa(...)` de su escenario elegido por su `grid_inicial` real:

```python
# En el Notebook 4, dentro de iniciar_partida():
grid = grid_inicial   # <- en vez de generar_mapa(esc["n"], esc["probabilidades"], esc["semilla"])
```

Así, todas las mecánicas, indicadores y escenarios que ya construyeron funcionan sobre el mapa real de su ciudad.


## 8. Sesión de mentoría individual por equipo

Preparen para la sesión de mentoría de hoy:

- Muestren su mapa derivado de NDVI real y expliquen qué representa cada zona.
- Compartan su ficha de proyecto y narrativa.
- Pregunten a su mentor: ¿el problema que eligieron es lo suficientemente específico? ¿falta algún dato importante de su zona?
- Anoten aquí (en una celda markdown nueva, si quieren) los comentarios de su mentor.


## 🧭 Bitácora de aprendizaje

**✏️ Completen cada punto como equipo:**

1. Regresen a su predicción de la Sección 0. Ahora que vieron sus datos reales, ¿qué les sorprendió más? *(reemplazar)*
2. Su videojuego ahora combina datos reales (NDVI) con reglas inventadas por ustedes (relaciones del Notebook 1, indicadores del Notebook 4). En sus palabras, ¿dónde está la línea entre "esto es dato real" y "esto es una simplificación que decidimos nosotros"? *(reemplazar)*
3. ¿Qué parte de la complejidad real de su ciudad se queda fuera de su videojuego, aunque usen datos satelitales reales? *(reemplazar)*
4. ¿Cómo cambiaría su narrativa si en vez de NDVI tuvieran datos de calidad del aire, tráfico o ingreso económico de su zona? *(reemplazar)*

## ✅ Producto esperado de esta sesión

- [ ] Datos satelitales reales (NDVI/uso de suelo) cargados y convertidos a mapa inicial del juego
- [ ] Checklist de integración de datos completado
- [ ] Zona y problema específico elegidos y documentados
- [ ] Narrativa definida (protagonista, meta, tensión, estética, nombre del juego)
- [ ] Archivo `proyecto_final_alpha.json` generado
- [ ] Bitácora de aprendizaje completa

**Mañana (Día 5):** pulir, testear y preparar la presentación final de su proyecto. 🎉
